## CIF/FOB ratio cases in direct-factor training cases

In [11]:
from pathlib import Path
import duckdb

path = Path(
    "/storage/vacoda/data/interim/cif_fob/target_h5/"
    "cif_fob_runs/cif_fob_upper_limit_1p457/"
    "target_h5_cif_fob_estimated_rates.parquet"
)

con = duckdb.connect()

result = con.execute(
    """
    WITH direct_training AS (
        SELECT
            import_value_cif,
            export_value_fob,
            import_value_cif / export_value_fob
                AS original_cif_fob_ratio

        FROM read_parquet(?)

        WHERE clean_estimation_eligible
          AND LOWER(CAST(factor_source AS VARCHAR)) = 'direct'
          AND import_value_cif IS NOT NULL
          AND isfinite(import_value_cif)
          AND import_value_cif > 0
          AND export_value_fob IS NOT NULL
          AND isfinite(export_value_fob)
          AND export_value_fob > 0
    ),

    classified AS (
        SELECT
            CASE
                WHEN original_cif_fob_ratio > 1.000001
                    THEN 'Importer CIF above exporter FOB'
                WHEN original_cif_fob_ratio < 0.999999
                    THEN 'Importer CIF below exporter FOB'
                ELSE 'Approximately equal'
            END AS direction,

            original_cif_fob_ratio

        FROM direct_training
    )

    SELECT
        direction,
        COUNT(*) AS rows,

        100.0 * COUNT(*) / SUM(COUNT(*)) OVER ()
            AS share,

        MEDIAN(original_cif_fob_ratio)
            AS median_original_ratio,

        QUANTILE_CONT(original_cif_fob_ratio, 0.10)
            AS ratio_p10,

        QUANTILE_CONT(original_cif_fob_ratio, 0.90)
            AS ratio_p90

    FROM classified

    GROUP BY direction

    ORDER BY
        CASE direction
            WHEN 'Importer CIF above exporter FOB' THEN 1
            WHEN 'Approximately equal' THEN 2
            ELSE 3
        END
    """,
    [str(path)],
).df()

print(result.to_string(index=False))

                      direction   rows     share  median_original_ratio  ratio_p10  ratio_p90
Importer CIF above exporter FOB 190624 54.058561               1.964222   1.064427  37.835682
            Approximately equal    163  0.046225               1.000000   1.000000   1.000000
Importer CIF below exporter FOB 161838 45.895214               0.418911   0.027719   0.920068


In [17]:
5487981 - 682692

4805289

In [13]:
result = con.execute(
    """
    SELECT
        COUNT(*) AS final_training_rows,

        COUNT(*) FILTER (
            WHERE import_value_cif IS NOT NULL
              AND isfinite(import_value_cif)
              AND import_value_cif > 0
              AND export_value_fob IS NOT NULL
              AND isfinite(export_value_fob)
              AND export_value_fob > 0
        ) AS rows_with_bilateral_values,

        COUNT(*) FILTER (
            WHERE import_value_cif IS NULL
               OR export_value_fob IS NULL
        ) AS rows_missing_at_least_one_bilateral_value,

        MEDIAN(observed_cif_fob_factor)
            AS median_training_target,

        MIN(observed_cif_fob_factor)
            AS minimum_training_target,

        MAX(observed_cif_fob_factor)
            AS maximum_training_target

    FROM read_parquet(?)

    WHERE clean_estimation_eligible
    """,
    [str(path)],
).df()

print(result.to_string(index=False))

 final_training_rows  rows_with_bilateral_values  rows_missing_at_least_one_bilateral_value  median_training_target  minimum_training_target  maximum_training_target
              606042                      352625                                     253417                1.069038                      1.0                 1.457336


In [14]:
columns = con.execute(
    """
    DESCRIBE
    SELECT *
    FROM read_parquet(?)
    """,
    [str(path)],
).df()

print(
    columns.loc[
        columns["column_name"].str.contains(
            "factor|direct|mirror|export|import|value",
            case=False,
            regex=True,
        ),
        ["column_name", "column_type"],
    ].to_string(index=False)
)

                                                  column_name column_type
                                                 exporter_iso     VARCHAR
                                                 importer_iso     VARCHAR
                                                factor_source     VARCHAR
                                      observed_cif_fob_factor      DOUBLE
                          import_direct_covered_primary_value      DOUBLE
                                             export_value_fob      DOUBLE
                                             import_value_cif      DOUBLE
                                      import_weight_estimated     BOOLEAN
                              lagged_pair_factor_observations      DOUBLE
                                     lagged_pair_factor_years      BIGINT
                                     lagged_pair_factor_found     BOOLEAN
                                          exporter_landlocked     INTEGER
                                      

In [15]:
result = con.execute(
    """
    SELECT
        CASE
            WHEN observed_cif_fob_factor > 1.000001
                THEN 'CIF above FOB'
            WHEN observed_cif_fob_factor < 0.999999
                THEN 'CIF below FOB'
            ELSE 'Approximately equal'
        END AS direction,

        COUNT(*) AS rows,

        100.0 * COUNT(*) / SUM(COUNT(*)) OVER ()
            AS share,

        MEDIAN(observed_cif_fob_factor)
            AS median_factor,

        QUANTILE_CONT(observed_cif_fob_factor, 0.10)
            AS factor_p10,

        QUANTILE_CONT(observed_cif_fob_factor, 0.90)
            AS factor_p90

    FROM read_parquet(?)

    WHERE clean_estimation_eligible
      AND LOWER(CAST(factor_source AS VARCHAR)) = 'direct'

    GROUP BY direction

    ORDER BY
        CASE direction
            WHEN 'CIF above FOB' THEN 1
            WHEN 'Approximately equal' THEN 2
            ELSE 3
        END
    """,
    [str(path)],
).df()

print(result.to_string(index=False))

          direction   rows     share  median_factor  factor_p10  factor_p90
      CIF above FOB 547226 90.295062       1.076406    1.019801    1.233335
Approximately equal  58816  9.704938       1.000000    1.000000    1.000000


In [ ]:
from pathlib import Path
import duckdb

path = Path(
    "/storage/vacoda/data/interim/cif_fob/target_h5/"
    "cif_fob_runs/cif_fob_upper_limit_1p457/"
    "comtrade_target_h5_cif_fob_corrected.parquet"
)

con = duckdb.connect()

In [9]:
result = con.execute(
    """
    WITH paired AS (
        SELECT
            import_value_cif,
            export_value_fob,
            import_value_cif / export_value_fob
                AS original_cif_fob_ratio

        FROM read_parquet(?)

        WHERE import_value_cif IS NOT NULL
          AND isfinite(import_value_cif)
          AND import_value_cif > 0
          AND export_value_fob IS NOT NULL
          AND isfinite(export_value_fob)
          AND export_value_fob > 0
    ),

    classified AS (
        SELECT
            CASE
                WHEN original_cif_fob_ratio > 1.000001
                    THEN 'Importer CIF above exporter FOB'
                WHEN original_cif_fob_ratio < 0.999999
                    THEN 'Importer CIF below exporter FOB'
                ELSE 'Approximately equal'
            END AS direction,

            original_cif_fob_ratio

        FROM paired
    )

    SELECT
        direction,
        COUNT(*) AS rows,
        100.0 * COUNT(*) / SUM(COUNT(*)) OVER ()
            AS share,

        MEDIAN(original_cif_fob_ratio)
            AS median_original_ratio,

        QUANTILE_CONT(original_cif_fob_ratio, 0.10)
            AS ratio_p10,

        QUANTILE_CONT(original_cif_fob_ratio, 0.90)
            AS ratio_p90

    FROM classified

    GROUP BY direction

    ORDER BY
        CASE direction
            WHEN 'Importer CIF above exporter FOB' THEN 1
            WHEN 'Approximately equal' THEN 2
            ELSE 3
        END
    """,
    [str(path)],
).df()

print(result.to_string(index=False))

                      direction    rows     share  median_original_ratio  ratio_p10  ratio_p90
Importer CIF above exporter FOB 1221000 52.454019               2.161780   1.069905  44.583994
            Approximately equal    1805  0.077543               1.000000   1.000000   1.000000
Importer CIF below exporter FOB 1104948 47.468438               0.407549   0.024046   0.919052


In [3]:
result = con.execute(
    """
    SELECT
        CASE
            WHEN cif_fob_correction_applied
                THEN 'Corrected'

            WHEN import_value_cif IS NULL
                THEN 'No importer CIF value'

            WHEN NOT isfinite(import_value_cif)
                THEN 'Importer CIF not finite'

            WHEN import_value_cif <= 0
                THEN 'Importer CIF non-positive'

            WHEN estimated_cif_fob_factor IS NULL
                THEN 'No estimated CIF/FOB factor'

            WHEN NOT isfinite(estimated_cif_fob_factor)
                THEN 'Estimated factor not finite'

            WHEN estimated_cif_fob_factor <= 0
                THEN 'Estimated factor non-positive'

            ELSE 'Other uncorrected reason'
        END AS reason,

        COUNT(*) AS rows

    FROM read_parquet(?)

    GROUP BY 1
    ORDER BY rows DESC
    """,
    [str(path)],
).df()

print(result.to_string(index=False))

                     reason    rows
                  Corrected 4022140
      No importer CIF value 1328738
No estimated CIF/FOB factor  137103


In [5]:
from pathlib import Path
import duckdb

training_path = Path(
    "/storage/vacoda/data/interim/cif_fob/target_h5/"
    "target_h5_cif_fob_training_features.parquet"
)

con = duckdb.connect()

result = con.execute(
    """
    SELECT
        pair_geography_found,
        initial_predictors_complete,
        prediction_eligible,
        COUNT(*) AS rows,
        COUNT(*) FILTER (
            WHERE import_value_cif IS NOT NULL
        ) AS rows_with_import_cif

    FROM read_parquet(?)

    GROUP BY
        pair_geography_found,
        initial_predictors_complete,
        prediction_eligible

    ORDER BY rows DESC
    """,
    [str(training_path)],
).df()

print(result.to_string(index=False))

BinderException: Binder Error: Referenced column "prediction_eligible" not found in FROM clause!
Candidate bindings: "direct_eligible", "training_eligible", "model_eligible", "model_eligibility_reason", "value_pair_eligible"

LINE 16:         prediction_eligible
                 ^

In [6]:
result = con.execute(
    """
    SELECT
        COUNT(*) AS total_bilateral_rows,

        COUNT(*) FILTER (
            WHERE pair_geography_found
        ) AS rows_with_pair_geography,

        COUNT(*) FILTER (
            WHERE NOT pair_geography_found
        ) AS rows_without_pair_geography,

        COUNT(*) FILTER (
            WHERE initial_predictors_complete
        ) AS rows_with_complete_initial_predictors,

        COUNT(*) FILTER (
            WHERE NOT pair_geography_found
              AND import_value_cif IS NOT NULL
        ) AS importer_rows_without_pair_geography

    FROM read_parquet(?)
    """,
    [str(training_path)],
).df()

print(result.to_string(index=False))

 total_bilateral_rows  rows_with_pair_geography  rows_without_pair_geography  rows_with_complete_initial_predictors  importer_rows_without_pair_geography
              5487981                   5301886                       186095                                5301886                                137103


In [7]:
result = con.execute(
    """
    WITH classified AS (
        SELECT
            CASE
                WHEN export_value_fob IS NULL
                    THEN 'Exporter FOB missing'
                WHEN NOT isfinite(export_value_fob)
                    THEN 'Exporter FOB non-finite'
                WHEN export_value_fob <= 0
                    THEN 'Exporter FOB zero or negative'
                ELSE 'Exporter FOB usable'
            END AS exporter_reason,

            CASE
                WHEN import_value_cif IS NULL
                    THEN 'Importer CIF missing'
                WHEN NOT isfinite(import_value_cif)
                    THEN 'Importer CIF non-finite'
                WHEN import_value_cif <= 0
                    THEN 'Importer CIF zero or negative'
                WHEN estimated_cif_fob_factor IS NULL
                    THEN 'No estimated CIF/FOB factor'
                WHEN NOT isfinite(estimated_cif_fob_factor)
                    THEN 'Estimated factor non-finite'
                WHEN estimated_cif_fob_factor <= 0
                    THEN 'Estimated factor zero or negative'
                WHEN import_value_fob_corrected IS NULL
                    THEN 'Corrected importer FOB missing'
                WHEN NOT isfinite(import_value_fob_corrected)
                    THEN 'Corrected importer FOB non-finite'
                WHEN import_value_fob_corrected <= 0
                    THEN 'Corrected importer FOB zero or negative'
                ELSE 'Corrected importer FOB usable'
            END AS importer_reason

        FROM read_parquet(?)
    )

    SELECT
        exporter_reason,
        importer_reason,
        COUNT(*) AS rows,
        100.0 * COUNT(*) / SUM(COUNT(*)) OVER ()
            AS share_of_neither_usable

    FROM classified

    WHERE exporter_reason <> 'Exporter FOB usable'
      AND importer_reason <> 'Corrected importer FOB usable'

    GROUP BY
        exporter_reason,
        importer_reason

    ORDER BY rows DESC
    """,
    [str(path)],
).df()

print(result.to_string(index=False))

              exporter_reason               importer_reason  rows  share_of_neither_usable
         Exporter FOB missing   No estimated CIF/FOB factor 90719                99.992284
Exporter FOB zero or negative          Importer CIF missing     4                 0.004409
         Exporter FOB missing Importer CIF zero or negative     3                 0.003307


## Example rows of corrected-trade-dat

In [20]:
from pathlib import Path
import duckdb
path = Path(
    "/storage/vacoda/data/interim/cif_fob/target_h5/cif_fob_runs/cif_fob_upper_limit_1p457/"
    "comtrade_target_h5_cif_fob_corrected.parquet"
)

In [21]:
examples = con.execute(
    """
    SELECT
        year,
        exporter_iso,
        importer_iso,

        export_value_fob,
        import_value_cif,
        estimated_cif_fob_factor,
        import_value_fob_corrected,

        import_value_cif / export_value_fob
            AS original_import_export_ratio,

        import_value_fob_corrected / export_value_fob
            AS corrected_import_export_ratio

    FROM read_parquet(?)

    WHERE export_value_fob IS NOT NULL
      AND isfinite(export_value_fob)
      AND export_value_fob > 0

      AND import_value_fob_corrected IS NOT NULL
      AND isfinite(import_value_fob_corrected)
      AND import_value_fob_corrected > 0

    ORDER BY random()
    LIMIT 20
    """,
    [str(path)],
).df()

examples

,year,exporter_iso,importer_iso,export_value_fob,import_value_cif,estimated_cif_fob_factor,import_value_fob_corrected,original_import_export_ratio,corrected_import_export_ratio
0,2025,LTU,NOR,12310.000,18630.729,1.064383,1.750378e+04,1.513463,1.421915
1,2019,CZE,AUS,10683.000,11256.626,1.110812,1.013369e+04,1.053695,0.948581
2,2019,KOR,SVN,1509.000,3067.820,1.098221,2.793444e+03,2.033015,1.851189
3,2020,DEU,MLT,7407.776,7136.440,1.060066,6.732069e+03,0.963371,0.908784
4,2019,IND,LUX,27290.192,12783.277,1.072181,1.192268e+04,0.468420,0.436885
5,2023,DEU,NOR,665.086,27387.804,1.058090,2.588420e+04,41.179342,38.918578
6,2017,ITA,SGP,602325.575,1409524.206,1.079042,1.306274e+06,2.340137,2.168718
7,2023,POL,EST,4828527.000,3412248.410,1.042670,3.272607e+06,0.706685,0.677765
8,2020,BRB,JAM,12131.000,12190.650,1.110192,1.098067e+04,1.004917,0.905174
9,2017,UKR,MDA,1911850.150,1955790.000,1.082773,1.806280e+06,1.022983,0.944781
